---
title: "TESTING SITE"
jupyter: python3
#code-fold: true
execute:
  echo: true
  output: asis
lightbox: 
  match: auto
  effect: fade
  desc-position: bottom
  loop: true
highlight-style: atom-one
code-tools: true
code-fold: true
---

In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON
from pathlib import Path
from collections import defaultdict

def query_WB(endpoint, query):

    # Initialize the wrapper
    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    # Fetch and parse results
    results = sparql.query().convert()

    return results 

def build_tree(results):
    # Map URLs to nodes and children
    nodes = {}
    children_map = defaultdict(list)
    for result in results["results"]["bindings"]:
        item_url = result["item"]["value"]
        item_label = result.get("itemLabel", {}).get("value", "")
        item_descr = result.get("itemDescription", {}).get("value", "")
        item_type = result.get("itemTypeLabel", {}).get("value", "")
        photo = result.get("photo", {}).get("value", "")
        creator = result.get("creator", {}).get("value", "")
        parent_url = result.get("parent", {}).get("value", "")

        # Node-Objekt initialisieren (keine Dopplungen bei Mehrfach-Fotos)
        if item_url not in nodes:
            nodes[item_url] = {
                "url": item_url,
                "label": item_label,
                "descr": item_descr,
                "type": item_type,
                "photos": [],
                "creators": set(),
                "parent": parent_url,
                "children": []
            }
        if photo and photo not in nodes[item_url]["photos"]:
            nodes[item_url]["photos"].append(photo)
        if creator:
            nodes[item_url]["creators"].add(creator)
        if parent_url:
            children_map[parent_url].append(item_url)

    # Kinder zu Eltern zuordnen
    for parent_url, child_list in children_map.items():
        for child_url in child_list:
            if parent_url in nodes and child_url in nodes:
                nodes[parent_url]["children"].append(nodes[child_url])

    # Wurzeln finden (keinen Parent ODER Parent ist nicht selbst ein Node)
    roots = [node for node in nodes.values() if not node["parent"] or node["parent"] not in nodes]
    return roots

def is_supported_image(path):
    ext = Path(path).suffix.lower()
    return ext in [".jpg", ".jpeg", ".png"]

def print_hierarchy(nodes, level=0):
    # Markdown/HTML/LaTeX-Ausgabe nach Bedarf hier gestalten
    indent = "#" * level + " "
    for node in nodes:
        print(f"""
{indent}* **{node['label']}** ({node['type']})
        """)
        if node["descr"]:
            print(f"""
            {indent}  _{node['descr']}_
            """)
        if node["url"]:
            print(f"""
            {indent}  [Wikibase]({node['url']})
            """)
        if node["photos"]:
            for photo in node["photos"]:
                if is_supported_image(photo):
                    print(f"{indent}  ![]({photo})")
                else:
                    print(f"{indent}  (Nicht unterstütztes Bildformat: {photo})")
        if node["creators"]:
            print(f"{indent}  Ersteller: {', '.join(node['creators'])}")
        if node["children"]:
            print_hierarchy(node["children"], level + 1)
        print()  # Abstand zwischen Einträgen






In [2]:
def make_book():

    endpoint = "https://query.kewl.org/sparql"

    query = """
PREFIX wd: <https://wikibase.kewl.org/entity/>
PREFIX wdt: <https://wikibase.kewl.org/prop/direct/>
PREFIX p: <https://wikibase.kewl.org/prop/>
PREFIX ps: <https://wikibase.kewl.org/prop/statement/>
PREFIX pq: <https://wikibase.kewl.org/prop/qualifier/>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX bd: <http://www.bigdata.com/rdf#>

SELECT DISTINCT ?item ?itemLabel ?itemDescription ?itemType ?itemTypeLabel ?photo ?creator ?parent ?parentLabel
WHERE {
  ?item wdt:P1 ?itemType .
  OPTIONAL { ?item wdt:P3 ?parent . }
  OPTIONAL {
    ?item p:P6 ?statement .
    ?statement ps:P6 ?photo .
    OPTIONAL { ?statement pq:P11 ?creator. }
  }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "de" }
}
ORDER BY ?item
LIMIT 100
    """


    results = query_WB(endpoint, query)
    tree = build_tree(results)
    print_hierarchy(tree)

make_book()


 * **Lüneburg, Kloster Lüne** (Bauwerk)
        

               _cps_deckenmalerei.db, 1_
            

               [Wikibase](https://wikibase.kewl.org/entity/Q10)
            
   ![](https://previous.bildindex.de/bilder/fmd10045419a.jpg)
   ![](https://previous.bildindex.de/bilder/fmd10045331a.jpg)
   ![](https://previous.bildindex.de/bilder/fmd10045330a.jpg)
   ![](https://previous.bildindex.de/bilder/fmd10045334a.jpg)
   ![](https://previous.bildindex.de/bilder/fmd10045371a.jpg)
   ![](https://previous.bildindex.de/bilder/fmd10045333a.jpg)
   Ersteller: Bunz, Achim

# * **Das Vorzimmer der Äbtissin** (Raum)
        

            #   _cps_deckenmalerei.db, 8_
            

            #   [Wikibase](https://wikibase.kewl.org/entity/Q11)
            
#   ![](https://previous.bildindex.de/bilder/fmd10045420a.jpg)
#   Ersteller: Bunz, Achim

## * **Die Deckenmalereireste** (Malerei)
        

            ##   _cps_deckenmalerei.db, 10_
            

            ##   [Wikibase](htt